# Run pipeline (01 → 03)

Runs notebooks **01–03**. Uses study paths from `config.yaml` when first-level beta maps
exist; otherwise generates `example_data/` and temporarily applies `config.example.yaml`.

In [1]:
import shutil
import subprocess
import sys
from pathlib import Path

import nbformat
import yaml
from nbconvert.preprocessors import ExecutePreprocessor

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
NB = PROJECT_ROOT / "notebooks"
CONFIG = PROJECT_ROOT / "config.yaml"
CONFIG_EXAMPLE = PROJECT_ROOT / "config.example.yaml"
CONFIG_BACKUP = PROJECT_ROOT / "config.yaml.pipeline_backup"
GEN_SCRIPT = PROJECT_ROOT / "example_data" / "generate_example_data.py"


def _resolve(path_str: str) -> Path:
    p = Path(path_str)
    return p.resolve() if p.is_absolute() else (PROJECT_ROOT / p).resolve()


def _study_data_ready() -> bool:
    cfg = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))
    fl = _resolve(cfg["paths"]["first_level_derivatives"]) / cfg["analysis"]["nii_str"]
    mask = _resolve(cfg["paths"]["brain_mask"])
    return fl.is_dir() and any(fl.glob("sub-*/patterns")) and mask.is_file()


use_example = not _study_data_ready()
restored = False
if use_example:
    print("Study first-level data not found — using example_data/.")
    subprocess.check_call([sys.executable, str(GEN_SCRIPT)], cwd=str(PROJECT_ROOT))
    if CONFIG.is_file():
        shutil.copy2(CONFIG, CONFIG_BACKUP)
    shutil.copy2(CONFIG_EXAMPLE, CONFIG)
else:
    print("Using study paths from config.yaml.")

try:
    steps = [
        "01_load_first_level_data.ipynb",
        "02_glm_and_contrasts.ipynb",
        "03_figure_panel_abc.ipynb",
    ]
    ep = ExecutePreprocessor(timeout=1800, kernel_name="python3")
    for s in steps:
        print(f"\n=== {s} ===")
        with open(NB / s, encoding="utf-8") as f:
            nb = nbformat.read(f, as_version=4)
        ep.preprocess(nb, {"metadata": {"path": str(NB)}})
        with open(NB / s, "w", encoding="utf-8") as f:
            nbformat.write(nb, f)
finally:
    if use_example and CONFIG_BACKUP.is_file():
        shutil.move(CONFIG_BACKUP, CONFIG)
        restored = True

print("\nDone. See results/ and figures/.")
if restored:
    print("Restored original config.yaml.")

Study first-level data not found — using example_data/.

=== 01_load_first_level_data.ipynb ===

=== 02_glm_and_contrasts.ipynb ===

=== 03_figure_panel_abc.ipynb ===

Done. See results/ and figures/.
Restored original config.yaml.
